# Kemiripan Gigi Depan — versi **DINOv3** (pembanding DINOv2)

Notebook ini **pipeline-nya sama persis** dengan `teeth_similarity_intro.ipynb`
(ROI crop -> embedding -> cosine ke 10 anchor -> predict_ac -> 3 probe), tapi encoder-nya
**DINOv3**. Tujuannya: banding angka langsung dengan DINOv2.

Jalankan **top-to-bottom (Run All)**. Butuh internet & akses HuggingFace (lihat catatan gated di bawah).

## 0. DINOv2 vs DINOv3 — apa bedanya (konsep)

| Aspek | DINOv2 | DINOv3 |
|---|---|---|
| Data pretrain | LVD-142M (~142 juta) | **LVD-1689M (~1,7 miliar)** |
| Patch size | /14 | **/16** |
| Inovasi kunci | self-distillation + iBOT | + **Gram anchoring** (fitur *dense*/patch lebih tajam, tahan training panjang) |
| Varian ViT | S/B/L/g | **S, S+, B, L, H+, 7B** (+ ConvNeXt) |
| Embedding dim (ViT-S) | 384 | 384 |
| Lisensi | **Apache-2.0** (bebas) | **DINOv3 License (gated)** — harus setujui terms di HuggingFace |

**Ekspektasi:** DINOv3 umumnya menghasilkan fitur lebih kuat & lebih transferable (terutama fitur
dense/patch dan resolusi tinggi). Untuk task kecil kita, bedanya bisa terasa di **separabilitas
anchor** dan **monotonisitas ordinal** (Probe 3). Tapi belum tentu besar — makanya kita ukur.

## 1. Setup & akses gated

Instalasi (sekali, di terminal):

```bash
pip install torch transformers pillow scikit-learn matplotlib scipy
```

**Bobot DINOv3 itu gated.** Sekali saja:
1. Buka halaman model, mis. https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m , klik **Agree/Access** untuk menyetujui lisensi.
2. Buat token di https://huggingface.co/settings/tokens , lalu di terminal: `huggingface-cli login` (tempel token).

Setelah itu `from_pretrained` bisa mengunduh bobotnya.

## 1b. CONFIG — parameter yang bisa diutak-atik (DINOv3)

Sama seperti notebook DINOv2, tapi encoder-nya DINOv3. **Default = benchmark DINOv3.**

| Param | Default | Opsi | Pengaruh |
|---|---|---|---|
| `encoder` | `dinov3-vits16` | `vits16` / `vitb16` / `vitl16` | Makin besar → fitur lebih kuat, lebih lambat |
| `pooling` | `cls` | `cls` / `mean` | `mean` = rata-rata patch token (buang CLS+register), sering lebih baik untuk beda halus |
| `temperature` | `0.1` | `0.1`–`0.3` | Kecil = tajam (argmax-like); besar = grade tertimbang lebih halus |
| `tta_flip` | `False` | `True`/`False` | `True` = embedding asli + flip dirata-ratakan → prediksi lebih stabil |
| `roi_s_thr` / `roi_v_thr` / `roi_pad` | `0.30 / 0.55 / 0.10` | — | Ambang & padding ROI gigi (identik dgn DINOv2) |

Catatan: resize/normalize input diurus otomatis oleh `AutoImageProcessor` DINOv3 (tak ada `img_size` manual).

In [ ]:
# ========= CONFIG (DINOv3) — default = benchmark DINOv3 =========
CONFIG = {
    "encoder":     "facebook/dinov3-vits16-pretrain-lvd1689m",  # vits16 | vitb16 | vitl16
    "pooling":     "cls",             # "cls"(baseline) | "mean"(rata-rata patch token)
    "temperature": 0.1,               # 0.1(tajam) .. 0.3(halus)
    "tta_flip":    False,             # True = TTA flip horizontal
    "roi_s_thr":   0.30,              # ambang saturasi ROI
    "roi_v_thr":   0.55,              # ambang brightness ROI
    "roi_pad":     0.10,              # padding ROI
}
print("CONFIG DINOv3:", CONFIG)

In [ ]:
import glob, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoImageProcessor, AutoModel

def pick_device():
    if torch.backends.mps.is_available(): return torch.device("mps")
    if torch.cuda.is_available():         return torch.device("cuda")
    return torch.device("cpu")
DEVICE = pick_device()
print("Device:", DEVICE)

# Ganti ke varian lain kalau mau: vitb16 / vitl16 (lebih besar, lebih kuat, lebih berat)
MODEL_ID = CONFIG["encoder"]

IMG_DIR = "Front Teeth Google"
paths = sorted(glob.glob(os.path.join(IMG_DIR, "front_google_*.png")))
names = [os.path.basename(p).replace("front_google_","").replace(".png","") for p in paths]
anchor_paths  = sorted(glob.glob("ac_references/ac_grade_*.png"))
anchor_grades = list(range(1, len(anchor_paths)+1))
print(len(paths), "foto uji |", len(anchor_paths), "anchor")

In [ ]:
# Load DINOv3 (butuh sudah login HuggingFace & setuju lisensi)
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).eval().to(DEVICE)
print("Loaded:", MODEL_ID)

## 2. ROI crop — **identik** dengan pipeline DINOv2

Fungsi crop gigi sama persis, supaya perbandingan adil: yang berubah hanya encoder-nya.

In [ ]:
def teeth_roi_box(rgb01, s_thr=None, v_thr=None, dens=0.12, pad=None):
    s_thr = CONFIG["roi_s_thr"] if s_thr is None else s_thr
    v_thr = CONFIG["roi_v_thr"] if v_thr is None else v_thr
    pad   = CONFIG["roi_pad"]   if pad   is None else pad
    mx = rgb01.max(-1); mn = rgb01.min(-1)
    v = mx; s = np.where(mx > 1e-6, (mx - mn) / (mx + 1e-6), 0.0)
    teeth = (s < s_thr) & (v > v_thr)
    H, W = teeth.shape
    col, row = teeth.mean(0), teeth.mean(1)
    def center_box(fw=0.8, fh=0.6):
        cw, ch = int(W*fw), int(H*fh); x0=(W-cw)//2; y0=(H-ch)//2
        return (x0, y0, x0+cw, y0+ch)
    if col.max() < 1e-3 or row.max() < 1e-3:
        return center_box()
    cx = np.where(col > dens*col.max())[0]; ry = np.where(row > dens*row.max())[0]
    x0, x1, y0, y1 = int(cx.min()), int(cx.max()), int(ry.min()), int(ry.max())
    bw, bh = x1-x0, y1-y0
    if bw < 0.25*W or bh < 0.15*H:
        return center_box()
    pw, ph = int(bw*pad), int(bh*pad)
    return (max(0,x0-pw), max(0,y0-ph), min(W,x1+pw), min(H,y1+ph))

def roi_pil(path):
    """ROI crop -> PIL (biarkan processor DINOv3 yang resize+normalize)."""
    im = Image.open(path).convert("RGB")
    box = teeth_roi_box(np.asarray(im, np.float32)/255.)
    return im.crop(box)

def roi_disp(path, size=224):
    return np.asarray(roi_pil(path).resize((size,size)), np.float32)/255.

@torch.no_grad()
def _feat3(pil_img):
    inputs = processor(images=pil_img, return_tensors="pt").to(DEVICE)
    out = model(**inputs)
    if CONFIG["pooling"] == "mean":
        n_reg = getattr(model.config, "num_register_tokens", 0)
        return out.last_hidden_state[0, 1+n_reg:].mean(0)          # patch tokens saja
    v = getattr(out, "pooler_output", None)
    return v[0] if v is not None else out.last_hidden_state[0, 0]  # CLS token

@torch.no_grad()
def dino3_embed(pil_img):
    f = _feat3(pil_img)
    if CONFIG["tta_flip"]:
        from PIL import ImageOps
        f = f + _feat3(ImageOps.mirror(pil_img))
    v = f.float().cpu().numpy()
    return v / (np.linalg.norm(v) + 1e-8)

def embed_path(path):       # helper: path -> embedding (ROI)
    return dino3_embed(roi_pil(path))

## 3. Similarity matrix (foto uji)

In [ ]:
dino = np.stack([embed_path(p) for p in paths])
print("Embeddings:", dino.shape)
S = cosine_similarity(dino)

plt.figure(figsize=(5.5,4.5))
plt.imshow(S, cmap="magma", vmin=S.min(), vmax=1)
plt.xticks(range(len(names)), names, rotation=45); plt.yticks(range(len(names)), names)
plt.colorbar(label="cosine"); plt.title(f"DINOv3 similarity ({MODEL_ID.split('/')[-1]})")
for i in range(len(names)):
    for j in range(len(names)):
        plt.text(j,i,f"{S[i,j]:.2f}",ha="center",va="center",
                 color="white" if S[i,j]<0.7 else "black",fontsize=7)
plt.tight_layout(); plt.show()

## 4. AC grader + test vs referensi paling mirip

In [ ]:
anchor_emb = np.stack([embed_path(p) for p in anchor_paths])
print("anchor embeddings:", anchor_emb.shape)

def predict_ac(img_path):
    q = embed_path(img_path)
    sims = cosine_similarity(q[None], anchor_emb)[0]
    hard = anchor_grades[int(np.argmax(sims))]
    w = np.exp(sims / CONFIG['temperature']); w /= w.sum()
    soft = float(np.dot(w, anchor_grades))
    return hard, soft, sims

print("\nPrediksi AC (DINOv3):")
for p, n in zip(paths, names):
    hard, soft, sims = predict_ac(p)
    top3 = np.argsort(-sims)[:3]
    print(f"  {n}: AC(1-NN)={hard:>2}  AC(weighted)={soft:4.1f}  top-3={[anchor_grades[i] for i in top3]}")

In [ ]:
# test vs referensi paling mirip
anchor_disp = [roi_disp(p) for p in anchor_paths]
test_disp   = [roi_disp(p) for p in paths]
TOPK = 3
fig, axes = plt.subplots(len(paths), 1+TOPK, figsize=(2.4*(1+TOPK), 2.4*len(paths)))
for i,(pth,n) in enumerate(zip(paths,names)):
    _, soft, sims = predict_ac(pth); order = np.argsort(-sims)[:TOPK]
    axes[i,0].imshow(test_disp[i]); axes[i,0].axis("off"); axes[i,0].set_title(f"TEST {n}\n(AC~{soft:.1f})",fontsize=9)
    for k,a in enumerate(order):
        axes[i,k+1].imshow(anchor_disp[a]); axes[i,k+1].set_xticks([]); axes[i,k+1].set_yticks([])
        tag = "paling mirip" if k==0 else f"#{k+1}"
        axes[i,k+1].set_title(f"{tag}\nAC {anchor_grades[a]}  sim={sims[a]:.2f}",fontsize=9,color=("green" if k==0 else "0.3"))
        if k==0:
            for sp in axes[i,k+1].spines.values(): sp.set_visible(True); sp.set_color("green"); sp.set_linewidth(2.5)
plt.tight_layout(); plt.show()

## 5. Probe: shape vs warna, & konsistensi ordinal (bandingkan dengan DINOv2)

Tiga probe yang sama. **Catat angkanya**, lalu banding dengan notebook DINOv2 di sel yang setara.

In [ ]:
# PROBE 1: grayscale (shape vs warna)
def roi_pil_gray(path):
    return roi_pil(path).convert("L").convert("RGB")

emb_c = np.stack([dino3_embed(roi_pil(p))      for p in paths])
emb_g = np.stack([dino3_embed(roi_pil_gray(p)) for p in paths])
anc_c = np.stack([dino3_embed(roi_pil(p))      for p in anchor_paths])
anc_g = np.stack([dino3_embed(roi_pil_gray(p)) for p in anchor_paths])
self_sim = np.mean([cosine_similarity(emb_c[i:i+1], emb_g[i:i+1])[0,0] for i in range(len(paths))])
pred_c = [anchor_grades[int(np.argmax(cosine_similarity(emb_c[i:i+1], anc_c)[0]))] for i in range(len(paths))]
pred_g = [anchor_grades[int(np.argmax(cosine_similarity(emb_g[i:i+1], anc_g)[0]))] for i in range(len(paths))]
agree = np.mean([a==b for a,b in zip(pred_c,pred_g)])
print(f"[DINOv3] cosine(warna,gray)={self_sim:.3f}  agreement grade={agree*100:.0f}%")

In [ ]:
# PROBE 2: white-balance jitter (tahan warna?)
def wb_jitter_pil(pil_img, rf, bf):
    a = np.asarray(pil_img, np.float32)/255.
    a[...,0]*=rf; a[...,2]*=bf; a=np.clip(a,0,1)
    return Image.fromarray((a*255).astype("uint8"))

jits = [(1.2,0.8),(0.8,1.2),(1.15,1.0),(1.0,0.85)]
drift=[]
for p in paths:
    base = roi_pil(p); eb = dino3_embed(base)
    sims = [cosine_similarity(eb[None], dino3_embed(wb_jitter_pil(base,rf,bf))[None])[0,0] for rf,bf in jits]
    drift.append(np.mean(sims))
print(f"[DINOv3] cosine(asli, warna-diubah)={np.mean(drift):.3f}  (->1 = tahan warna)")

In [ ]:
# PROBE 3: monotonisitas ordinal anchor (Spearman)
from scipy.stats import spearmanr
A = np.stack([embed_path(p) for p in anchor_paths])
Sm = cosine_similarity(A); D = 1 - Sm
iu = np.triu_indices(len(anchor_grades),1)
gap  = np.abs(np.subtract.outer(anchor_grades,anchor_grades))[iu]
dist = D[iu]
rho, pval = spearmanr(gap, dist)
print(f"[DINOv3] Spearman(|dgrade|, jarak)= {rho:.3f}  (p={pval:.3g})")

fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].imshow(Sm,cmap="magma"); ax[0].set_title("Similarity antar-anchor (DINOv3)")
ax[0].set_xticks(range(10)); ax[0].set_xticklabels(anchor_grades); ax[0].set_yticks(range(10)); ax[0].set_yticklabels(anchor_grades)
ax[1].scatter(gap,dist,alpha=0.6); ax[1].set_xlabel("|dgrade|"); ax[1].set_ylabel("jarak (1-cos)")
ax[1].set_title(f"Ordinalitas (rho={rho:.2f})")
plt.tight_layout(); plt.show()

## 5b. Evaluasi weak-label (kompas tuning)

Sama seperti DINOv2. **Hanya bermakna kalau dataset punya nama file deskriptif** (mis. set drg Laura).
Untuk set Google (`front_google_*`) skornya seragam, jadi dilewati otomatis.

In [ ]:
from scipy.stats import spearmanr
SEVERITY = [("normal",1),("diastema",3),("open bite",5),("openbite",5),
            ("crowding",5),("midline",5),("crossbite",6),("deep bite",6),
            ("protrusion",6),("kelas 3",8)]
def weak_severity(name):
    n=name.lower(); sc=1
    for kw,w in SEVERITY:
        if kw in n: sc=max(sc,w)
    return sc
weak = np.array([weak_severity(n) for n in names])
if len(set(weak.tolist())) < 2:
    print("Dataset ini tak punya label-nama deskriptif -> weak-eval dilewati.")
    print("(Arahkan ke set klinis spt 'Front Teeth drg Laura' untuk memakainya.)")
else:
    preds = np.array([predict_ac(p)[1] for p in paths])
    rho,pval = spearmanr(weak, preds)
    print(f"[DINOv3] Spearman(keparahan-nama, prediksi AC) = {rho:.3f}  (p={pval:.3g})")
    for i in np.argsort(weak):
        print(f"  weak={weak[i]}  AC_pred={preds[i]:4.1f}   {names[i][:48]}")

## 6. Cara membandingkan dengan DINOv2

Jalankan kedua notebook, lalu bandingkan angka yang setara:

- **Probe 3 (Spearman rho)** — paling penting. Makin tinggi = ruang embedding makin "paham" urutan
  keparahan AC. Encoder dengan rho lebih tinggi biasanya jadi grader lebih baik.
- **Separabilitas anchor** — lihat heatmap antar-anchor: pola gradien/band yang rapi = grade
  terpisah baik. Blok seragam = grade sulit dibedakan.
- **Probe 1 & 2** — seberapa shape-based / tahan warna. Idealnya keduanya tinggi.
- **predict_ac** — apakah grade untuk foto yang sama lebih masuk akal di DINOv2 atau DINOv3?

Catatan adil: bedanya bisa kecil pada dataset sekecil ini. Kesimpulan kuat tetap butuh set
validasi ber-grade dokter (MAE / within-±1) untuk kedua encoder.

**Trade-off lisensi:** DINOv2 = Apache-2.0 (bebas, mudah deploy). DINOv3 = gated (perlu setuju
terms). Kalau kualitasnya setara untuk task-mu, DINOv2 lebih ringan secara legal untuk on-device.